# Framework final controlado — CNN/RAG/SLM para diagnóstico de motores por termografia

Notebook único para executar **Mistral-7B**, **Qwen2.5-3B**, **TinyLlama-1.1B** e **Zephyr-7B** sequencialmente, sob o mesmo protocolo. Cada modelo é carregado, executado, salvo e descarregado antes do próximo.


> **Revisão 1:** corrigidas referências de configuração para `GLOBAL_CONFIG`. A geração foi mantida sem regeneração automática; a auditoria apenas classifica as saídas.


## 1. Instalação

Execute esta célula, reinicie o runtime e continue a partir da seção 2.

In [ ]:
# ============================================================
# CÉLULA 1 — INSTALAÇÃO FIXADA
# ============================================================
!pip install -q \
  pandas==2.2.2 \
  transformers==4.45.2 \
  accelerate==0.34.2 \
  bitsandbytes==0.49.2 \
  sentence-transformers==3.1.1 \
  evaluate==0.4.3 \
  rouge-score==0.1.2 \
  nltk==3.9.1 \
  scikit-learn==1.5.2 \
  faiss-cpu \
  sentencepiece \
  huggingface_hub \
  tqdm

In [ ]:
# ============================================================
# CÉLULA 1.1 — REINICIAR RUNTIME
# ============================================================
# Execute uma vez depois da instalação.
import os
os.kill(os.getpid(), 9)

## 2. Imports e verificação do ambiente

In [ ]:

import os
import json
import time
import gc
import textwrap
import hashlib
import platform
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)
from sentence_transformers import SentenceTransformer
import faiss

print("torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Configuração central do experimento

A lista `MODELS_TO_RUN` controla os modelos executados. Por padrão, os quatro são processados em sequência.


In [ ]:
# ============================================================
# CONFIGURAÇÃO FINAL PADRONIZADA — V4 MULTIMODELO + AUDITORIA
# Executa todos os modelos sequencialmente no mesmo runtime.
# ============================================================

MODELS_TO_RUN = ["mistral", "qwen", "tinyllama", "zephyr"]

GLOBAL_CONFIG = {
    "experiment_version": "final_controlled_v4_multimodel_audited",
    "embedding_model": "sentence-transformers/all-mpnet-base-v2",
    "rag_json_path": "/content/RAG_motores_eletricos.json",
    "cnn_predictions_path": "/content/holdout_test_predictions.csv",
    "rag_top_k": 3,

    # Limite comum aos quatro modelos.
    "max_input_tokens": 1280,
    "prompt_safety_margin": 32,
    "minimum_context_tokens": 240,
    "results_dir": "/content/results_slm_rag_final_v4",

    # Parâmetros iguais para todos os modelos.
    "max_new_tokens": 700,
    "do_sample": True,
    "temperature": 0.2,
    "top_p": 0.9,
    "top_k": None,
    "repetition_penalty": 1.1,

    # Reprodutibilidade.
    "base_seed": 42,

    # Controle de qualidade.
    "minimum_report_chars": 120,

    # A rodada não é interrompida por falhas generativas.
    # Elas são preservadas e classificadas no CSV.
    "stop_on_model_error": False,
}

MODEL_PROFILES = {
    "mistral": {
        "model_key": "mistral",
        "model_name": "mistralai/Mistral-7B-Instruct-v0.2",
        "model_label": "Mistral-7B-Instruct-v0.2",
        "loading_mode": "4bit_nf4",
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": False,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "mistral",
        "notes": "Mistral validado em 4-bit NF4; double_quant=False.",
    },
    "qwen": {
        "model_key": "qwen",
        "model_name": "Qwen/Qwen2.5-3B-Instruct",
        "model_label": "Qwen2.5-3B-Instruct-FP16",
        "loading_mode": "fp16",
        "load_in_4bit": False,
        "torch_dtype": "float16",
        "trust_remote_code": True,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "qwen",
        "notes": "Qwen validado em FP16.",
    },
    "tinyllama": {
        "model_key": "tinyllama",
        "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "model_label": "TinyLlama-1.1B-Chat-v1.0-FP16",
        "loading_mode": "fp16",
        "load_in_4bit": False,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": "left",
        "invalid_rules": "tinyllama",
        "notes": "TinyLlama em FP16; limitações generativas são auditadas, não corrigidas.",
    },
    "zephyr": {
        "model_key": "zephyr",
        "model_name": "HuggingFaceH4/zephyr-7b-beta",
        "model_label": "Zephyr-7B-Beta-4bit",
        "loading_mode": "4bit_nf4",
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": True,
        "torch_dtype": "float16",
        "trust_remote_code": False,
        "use_fast_tokenizer": True,
        "padding_side": None,
        "invalid_rules": "generic",
        "notes": "Zephyr em 4-bit NF4; double_quant=True.",
    },
}

invalid_models = [m for m in MODELS_TO_RUN if m not in MODEL_PROFILES]
if invalid_models:
    raise ValueError(f"Modelos inválidos em MODELS_TO_RUN: {invalid_models}")

results_dir = Path(GLOBAL_CONFIG["results_dir"])
results_dir.mkdir(parents=True, exist_ok=True)

print("Modelos programados:", MODELS_TO_RUN)
print("Diretório de resultados:", results_dir)
print(json.dumps(GLOBAL_CONFIG, indent=2, ensure_ascii=False))


## 4. Upload/carregamento do JSON do RAG

In [ ]:
from google.colab import files

rag_path = Path(GLOBAL_CONFIG["rag_json_path"])

if not rag_path.exists():
    print("Envie o arquivo RAG_motores_eletricos.json")
    uploaded = files.upload()
    for name in uploaded.keys():
        if name.endswith(".json"):
            Path(name).rename(rag_path)
            print(f"Arquivo salvo em: {rag_path}")
else:
    print(f"Arquivo encontrado: {rag_path}")

In [ ]:
def load_rag_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

rag_data = load_rag_json(GLOBAL_CONFIG["rag_json_path"])
failures = rag_data["document"]["failures"]

print(f"Total de classes/falhas no RAG: {len(failures)}")
print([item.get("type") for item in failures])

## 5. Criação dos documentos do RAG

In [ ]:
# ============================================================
# DOCUMENTOS RAG — 6 ENTRADAS ORIGINAIS (UMA POR CLASSE)
# A base não é dividida em 18 chunks. O limite é aplicado somente
# ao contexto efetivamente enviado ao modelo.
# ============================================================
def as_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                parts.append("; ".join(f"{k}: {v}" for k, v in item.items()))
            else:
                parts.append(str(item))
        return "; ".join(parts)
    if isinstance(value, dict):
        return " | ".join(
            f"{key}: {as_text(item)}"
            for key, item in value.items()
        )
    return str(value)


documents = []

for fault in failures:
    fault_type = fault.get("type", "")

    parts = [
        f"Fault Type: {fault_type}",
        f"Aliases: {as_text(fault.get('aliases', []))}",
        f"Description: {as_text(fault.get('description'))}",
        f"Physical mechanism: {as_text(fault.get('physical_mechanism'))}",
        f"Root causes: {as_text(fault.get('root_causes'))}",
        f"Observable evidence: {as_text(fault.get('observable_evidence'))}",
        f"System effects: {as_text(fault.get('system_effects'))}",
        f"Maintenance actions: {as_text(fault.get('maintenance_actions'))}",
        f"Diagnostic checks: {as_text(fault.get('diagnostic_checks'))}",
        f"Risk level: {as_text(fault.get('risk_level'))}",
        f"Standards and guidelines: {as_text(fault.get('standards_and_guidelines'))}",
        f"RAG retrieval notes: {as_text(fault.get('rag_retrieval_notes'))}",
    ]

    text = "\n".join(p for p in parts if not p.endswith(": "))

    documents.append({
        "fault_type": fault_type,
        "section": "full_entry",
        "risk_level": fault.get("risk_level", ""),
        "text": text,
    })

print("Total de documentos para recuperação:", len(documents))
assert len(documents) == len(failures), "Esperava-se uma entrada RAG por falha."
print(documents[0]["text"][:1800])


## 6. Embeddings + FAISS

In [ ]:
embedding_model = SentenceTransformer(GLOBAL_CONFIG["embedding_model"])
texts = [doc["text"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Índice FAISS criado:", index.ntotal)

## 7. Recuperação RAG

In [ ]:

CLASS_ALIASES = {
    "lack of phase": "Phase Loss",
    "phase loss": "Phase Loss",
    "normal": "Normal Operation",
    "normal operation": "Normal Operation",
    "bearing failure": "Bearing Failure",
    "blocked rotor": "Blocked Rotor",
    "overheating": "Overheating",
    "ventilation defect": "Ventilation Defect",
}

def canonicalize_class(name: str) -> str:
    key = str(name).strip().lower()
    return CLASS_ALIASES.get(key, str(name).strip())


def build_query(predicted_class: str, confidence: float, observed_evidence: str = "") -> str:
    canonical_class = canonicalize_class(predicted_class)
    return (
        f"Predicted fault: {canonical_class}. "
        f"CNN confidence: {confidence:.3f}. "
        f"Observed evidence: {observed_evidence}"
    )


def retrieve_context(
    predicted_class: str,
    confidence: float,
    observed_evidence: str = "",
    top_k: int = 3
) -> List[Dict[str, Any]]:

    query = build_query(predicted_class, confidence, observed_evidence)
    query_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_emb, top_k)

    retrieved = []
    for score, idx in zip(scores[0], indices[0]):
        doc = dict(documents[int(idx)])
        doc["score"] = float(score)
        retrieved.append(doc)

    return retrieved


## 8. Prompt técnico padronizado

In [ ]:
def token_count(text: str, add_special_tokens: bool = False) -> int:
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=add_special_tokens
        )
    )


def trim_to_token_budget(text: str, token_budget: int) -> str:
    """Recorta pelo tokenizer do modelo, sem exceder o orçamento."""
    if token_budget <= 0:
        return ""

    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False
    )

    if len(token_ids) <= token_budget:
        return str(text).strip()

    return tokenizer.decode(
        token_ids[:token_budget],
        skip_special_tokens=True
    ).strip()


def format_context_block(
    position: int,
    doc: Dict[str, Any],
    body_text: str
) -> str:
    return (
        f"[Context {position}]\n"
        f"Fault type: {doc.get('fault_type', 'NA')}\n"
        f"Section: {doc.get('section', 'NA')}\n"
        f"Retrieval score: {doc.get('score', 0.0):.4f}\n"
        f"{body_text}"
    )


# def prompt_from_context(
#     canonical_class: str,
#     confidence: float,
#     evidence_text: str,
#     context_text: str
# ) -> str:
#     prompt = f"""
# You are an industrial maintenance specialist in induction motor fault diagnosis using infrared thermography.

# CNN output:
# - Predicted fault: {canonical_class}
# - CNN confidence: {confidence:.3f}

# Observed thermal/operational evidence:
# {evidence_text}

# Retrieved technical context from the RAG knowledge base:
# {context_text}

# Write one complete and concise technical diagnostic report in English.

# Required sections:
# 1. Technical diagnosis
# 2. Supporting evidence
# 3. Probable causes
# 4. Recommended maintenance actions
# 5. Severity and risk assessment

# Mandatory rules:
# - Use only information supported by the CNN output and retrieved context.
# - Clearly distinguish direct CNN outputs from probable or inferred causes.
# - Do not claim that the CNN observed textual thermal evidence when none was produced.
# - Do not invent measurements, standards, citations, or inspection results.
# - Prefer synthesis over copying long passages from the context.
# - Keep the complete report between approximately 250 and 450 words.
# - Finish all five sections and end with a complete sentence.
# """
#     return textwrap.dedent(prompt).strip()

def prompt_from_context(
    canonical_class: str,
    confidence: float,
    evidence_text: str,
    context_text: str
) -> str:

    prompt = f"""
You are an industrial maintenance specialist in induction motor fault diagnosis using infrared thermography.

CNN prediction:
- Predicted fault: {canonical_class}
- Confidence: {confidence:.3f}

Observed evidence:
{evidence_text}

Retrieved technical knowledge:
{context_text}

Generate ONE concise technical diagnostic report in English.

The report MUST contain EXACTLY the following five sections:

1. Technical diagnosis
2. Supporting evidence
3. Probable causes
4. Recommended maintenance actions
5. Severity and risk assessment

Mandatory rules:

- Use ONLY information supported by the CNN prediction and the retrieved context.
- Clearly distinguish confirmed information from probable causes.
- Never invent measurements, temperatures, standards, inspections or evidence.
- Do not state that the CNN observed textual thermal patterns unless they are explicitly provided.
- Summarize instead of copying the retrieved context.
- Write exactly five sections using ONLY the headings above.
- Use no more than THREE short sentences per section.
- Keep the entire report between approximately 180 and 300 words.
- Do NOT write introductions, conclusions, summaries, notes, bullet lists or additional sections.
- Do NOT repeat information across sections.
- Finish immediately after completing Section 5.
- End with one complete sentence.

Produce only the report.
"""

    return textwrap.dedent(prompt).strip()


def build_prompt(
    predicted_class: str,
    confidence: float,
    retrieved_docs: List[Dict[str, Any]],
    observed_evidence: str = ""
):
    """
    Mantém os 6 documentos originais no FAISS e top-k=3.
    Limita somente o texto dos documentos recuperados, usando o tokenizer
    do modelo selecionado e verificando o tamanho APÓS o chat template.

    Retorna:
      prompt_final,
      contexto_exato_fornecido,
      metadados_do_orçamento.
    """
    canonical_class = canonicalize_class(predicted_class)

    evidence_text = (
        str(observed_evidence).strip()
        if str(observed_evidence).strip()
        else "No textual thermal evidence was produced by the CNN; only the predicted class and confidence are available."
    )

    empty_prompt = prompt_from_context(
        canonical_class,
        confidence,
        evidence_text,
        context_text=""
    )
    empty_chat = apply_model_chat_template(empty_prompt)
    fixed_tokens = token_count(empty_chat, add_special_tokens=False)

    max_input_tokens = int(GLOBAL_CONFIG["max_input_tokens"])
    safety_margin = int(GLOBAL_CONFIG.get("prompt_safety_margin", 32))
    available_context_tokens = max_input_tokens - fixed_tokens - safety_margin

    minimum_context_tokens = int(
        GLOBAL_CONFIG.get("minimum_context_tokens", 240)
    )
    if available_context_tokens < minimum_context_tokens:
        raise RuntimeError(
            "Orçamento insuficiente para contexto: "
            f"{available_context_tokens} tokens disponíveis."
        )

    # Distribuição igual entre os top-k para não favorecer a posição do documento.
    per_document_budget = max(
        1,
        available_context_tokens // max(1, len(retrieved_docs))
    )

    blocks = []
    context_metadata = []

    for i, doc in enumerate(retrieved_docs, start=1):
        header = format_context_block(i, doc, body_text="")
        header_tokens = token_count(header, add_special_tokens=False)
        body_budget = max(1, per_document_budget - header_tokens)

        original_text = str(doc.get("text", ""))
        compact_text = trim_to_token_budget(original_text, body_budget)
        block = format_context_block(i, doc, compact_text)
        blocks.append(block)

        context_metadata.append({
            "rank": i,
            "fault_type": doc.get("fault_type", "NA"),
            "section": doc.get("section", "NA"),
            "retrieval_score": float(doc.get("score", 0.0)),
            "original_document_tokens": token_count(
                original_text,
                add_special_tokens=False
            ),
            "provided_document_tokens": token_count(
                compact_text,
                add_special_tokens=False
            ),
            "document_was_trimmed": compact_text.strip() != original_text.strip(),
        })

    context_text = "\n\n".join(blocks)
    prompt = prompt_from_context(
        canonical_class,
        confidence,
        evidence_text,
        context_text
    )

    # Ajuste final determinístico, considerando exatamente o chat template.
    chat_text = apply_model_chat_template(prompt)
    final_tokens = token_count(chat_text, add_special_tokens=False)
    target = max_input_tokens - safety_margin

    while final_tokens > target:
        # Retira um pequeno bloco de tokens do documento atualmente mais longo.
        longest_idx = max(
            range(len(context_metadata)),
            key=lambda idx: context_metadata[idx]["provided_document_tokens"]
        )
        current = context_metadata[longest_idx]["provided_document_tokens"]
        if current <= 24:
            raise RuntimeError(
                "Não foi possível reduzir o prompt ao limite sem remover "
                "praticamente todo o contexto."
            )

        new_budget = max(24, current - 16)
        doc = retrieved_docs[longest_idx]
        compact_text = trim_to_token_budget(
            str(doc.get("text", "")),
            new_budget
        )
        blocks[longest_idx] = format_context_block(
            longest_idx + 1,
            doc,
            compact_text
        )
        context_metadata[longest_idx]["provided_document_tokens"] = (
            token_count(compact_text, add_special_tokens=False)
        )
        context_metadata[longest_idx]["document_was_trimmed"] = True

        context_text = "\n\n".join(blocks)
        prompt = prompt_from_context(
            canonical_class,
            confidence,
            evidence_text,
            context_text
        )
        chat_text = apply_model_chat_template(prompt)
        final_tokens = token_count(chat_text, add_special_tokens=False)

    budget_metadata = {
        "max_input_tokens": max_input_tokens,
        "safety_margin": safety_margin,
        "fixed_prompt_tokens": fixed_tokens,
        "available_context_tokens": available_context_tokens,
        "final_chat_input_tokens": final_tokens,
        "documents": context_metadata,
    }

    return prompt, context_text, budget_metadata


In [ ]:
# ============================================================
# UPLOAD DO CSV DE SAÍDA DA CNN (HOLD-OUT)
# ============================================================

from google.colab import files
import pandas as pd
from pathlib import Path

print("Selecione o arquivo holdout_test_predictions.csv")

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("Nenhum arquivo foi enviado.")

csv_name = list(uploaded.keys())[0]

# Caminho que será utilizado em todo o notebook
CNN_PREDICTIONS_PATH = Path("/content/holdout_test_predictions.csv")

# Renomeia automaticamente para manter o restante do código igual
Path(csv_name).rename(CNN_PREDICTIONS_PATH)

print(f"Arquivo salvo em: {CNN_PREDICTIONS_PATH}")

# Carrega para conferência
holdout_df = pd.read_csv(CNN_PREDICTIONS_PATH)

print("\nArquivo carregado com sucesso!")
print(f"Número de amostras: {len(holdout_df)}")
print(f"Número de colunas: {len(holdout_df.columns)}")

display(holdout_df.head())

## 9. Casos de teste padronizados

In [ ]:
# ============================================================
# CASOS REAIS DA CNN — SELEÇÃO AUTOMÁTICA NO HOLD-OUT
# Uma amostra corretamente classificada por classe, cuja confiança
# seja a mais próxima da mediana da própria classe.
# ============================================================
CNN_CSV_PATH = GLOBAL_CONFIG["cnn_predictions_path"]

if not Path(CNN_CSV_PATH).exists():
    raise FileNotFoundError(
        f"Arquivo da CNN não encontrado: {CNN_CSV_PATH}. "
        "Envie holdout_test_predictions.csv para o Colab."
    )

holdout_df = pd.read_csv(CNN_CSV_PATH)

COLUMN_CANDIDATES = {
    "true_class": ["true_class", "true_label", "actual_class", "target_class"],
    "predicted_class": [
        "predicted_class", "pred_class", "prediction", "predicted_label"
    ],
    "confidence": [
        "predicted_confidence", "confidence", "max_probability", "probability"
    ],
    "sample_id": [
        "sample_id", "image_id", "filename", "file_name", "path", "index"
    ],
}

def resolve_column(df, candidates, required=True):
    for name in candidates:
        if name in df.columns:
            return name
    if required:
        raise KeyError(
            f"Nenhuma das colunas esperadas foi encontrada: {candidates}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    return None

true_col = resolve_column(holdout_df, COLUMN_CANDIDATES["true_class"])
pred_col = resolve_column(holdout_df, COLUMN_CANDIDATES["predicted_class"])
conf_col = resolve_column(holdout_df, COLUMN_CANDIDATES["confidence"])
id_col = resolve_column(
    holdout_df,
    COLUMN_CANDIDATES["sample_id"],
    required=False
)

working_df = holdout_df.copy()
working_df["_true_canonical"] = working_df[true_col].map(canonicalize_class)
working_df["_pred_canonical"] = working_df[pred_col].map(canonicalize_class)
working_df["_confidence"] = pd.to_numeric(
    working_df[conf_col],
    errors="coerce"
)

working_df = working_df[
    working_df["_true_canonical"] == working_df["_pred_canonical"]
].dropna(subset=["_confidence"])

selected_rows = []

for class_name, group in working_df.groupby("_true_canonical", sort=True):
    median_confidence = group["_confidence"].median()
    chosen = (
        group.assign(
            _distance_to_median=(
                group["_confidence"] - median_confidence
            ).abs()
        )
        .sort_values(
            ["_distance_to_median", "_confidence"],
            ascending=[True, True]
        )
        .iloc[0]
    )
    selected_rows.append(chosen)

selected_cases_df = pd.DataFrame(selected_rows).reset_index(drop=True)

test_cases = []
for row_number, row in selected_cases_df.iterrows():
    raw_id = (
        str(row[id_col])
        if id_col is not None
        else f"holdout_row_{int(row.name)}"
    )

    test_cases.append({
        "sample_id": raw_id,
        "true_class": row["_true_canonical"],
        "predicted_class": row["_pred_canonical"],
        "confidence": float(row["_confidence"]),

        # A CNN classifica a imagem, mas não gera descrição textual da termografia.
        # Mantemos vazio para não introduzir evidência manual.
        "observed_evidence": "",

        # Rastreabilidade
        "cnn_source_row": int(row.name),
        "selection_rule": "correct_prediction_closest_to_class_median_confidence",
    })

expected_classes = set(working_df["_true_canonical"].unique())
selected_classes = {case["predicted_class"] for case in test_cases}

if selected_classes != expected_classes:
    raise RuntimeError(
        "A seleção não representou todas as classes corretamente classificadas. "
        f"Esperadas: {sorted(expected_classes)}; "
        f"selecionadas: {sorted(selected_classes)}"
    )

print(
    f"Hold-out: {len(holdout_df)} amostras | "
    f"corretas: {len(working_df)} | "
    f"casos selecionados: {len(test_cases)}"
)
display(pd.DataFrame(test_cases))


## 10. Adaptador único de carregamento do SLM

Esta célula preserva as diferenças críticas de cada modelo.

In [ ]:
def dtype_from_string(dtype_name: str):
    if dtype_name == "bfloat16":
        return torch.bfloat16
    if dtype_name == "float32":
        return torch.float32
    return torch.float16


def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def unload_slm():
    """Descarrega o modelo atual antes de carregar o próximo."""
    global tokenizer, model

    if "model" in globals():
        try:
            del model
        except Exception:
            pass

    if "tokenizer" in globals():
        try:
            del tokenizer
        except Exception:
            pass

    clear_cuda()


def load_slm(config: Dict[str, Any]):
    clear_cuda()

    model_name = config["model_name"]
    torch_dtype = dtype_from_string(config.get("torch_dtype", "float16"))

    tokenizer_local = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=config.get("use_fast_tokenizer", True),
        trust_remote_code=config.get("trust_remote_code", False),
    )

    if tokenizer_local.pad_token is None:
        tokenizer_local.pad_token = tokenizer_local.eos_token

    if config.get("padding_side") is not None:
        tokenizer_local.padding_side = config["padding_side"]

    # Ajuste específico do Qwen, baseado na configuração e não em variável global.
    if config.get("model_key") == "qwen":
        tokenizer_local.pad_token = tokenizer_local.eos_token
        tokenizer_local.pad_token_id = tokenizer_local.eos_token_id

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch_dtype,
        "trust_remote_code": config.get("trust_remote_code", False),
    }

    if config.get("load_in_4bit", False):
        compute_dtype = dtype_from_string(
            config.get("bnb_4bit_compute_dtype", "float16")
        )
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type=config.get("bnb_4bit_quant_type", "nf4"),
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=config.get(
                "bnb_4bit_use_double_quant", False
            ),
        )
        model_kwargs["quantization_config"] = bnb_config

    model_local = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs,
    )
    model_local.eval()

    model_local.config.pad_token_id = tokenizer_local.pad_token_id
    model_local.generation_config.pad_token_id = tokenizer_local.pad_token_id
    model_local.generation_config.eos_token_id = tokenizer_local.eos_token_id

    print("Modelo carregado:", model_name)
    print("Label:", config.get("model_label"))
    print("Modo:", config.get("loading_mode"))
    print("Observação:", config.get("notes"))
    print("Chat template existe?", tokenizer_local.chat_template is not None)

    if torch.cuda.is_available():
        print(
            "VRAM alocada:",
            round(torch.cuda.memory_allocated() / 1024**3, 3),
            "GB",
        )
        print(
            "VRAM reservada:",
            round(torch.cuda.memory_reserved() / 1024**3, 3),
            "GB",
        )

    return tokenizer_local, model_local


print("Funções de carregamento prontas. Os modelos serão carregados na execução multimodelo.")


## 11. Geração padronizada com regras específicas por modelo

In [ ]:
import re


FAULT_CANONICAL_MAP = {
    "normal": "normal operation",
    "normal operation": "normal operation",
    "normal condition": "normal operation",
    "bearing failure": "bearing failure",
    "bearing fault": "bearing failure",
    "bearing defect": "bearing failure",
    "blocked rotor": "blocked rotor",
    "rotor blocked": "blocked rotor",
    "locked rotor": "blocked rotor",
    "phase loss": "phase loss",
    "lack of phase": "phase loss",
    "loss of phase": "phase loss",
    "phase failure": "phase loss",
    "single phasing": "phase loss",
    "overheating": "overheating",
    "ventilation defect": "ventilation defect",
    "ventilation failure": "ventilation defect",
    "cooling failure": "ventilation defect",
}


def normalize_fault_name(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None

    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9\s-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    # Remove complementos comuns depois da classe.
    text = re.split(
        r"\s+(?:with|characterized|indicated|detected|observed|condition)\b",
        text,
        maxsplit=1,
    )[0].strip()

    return FAULT_CANONICAL_MAP.get(text, text)


def extract_declared_fault(report: str) -> Optional[str]:
    """Extrai somente declarações explícitas de diagnóstico."""
    patterns = [
        r"predicted fault\s*:\s*([^\n.;]+)",
        r"fault type\s*:\s*([^\n.;]+)",
        r"diagnosed fault\s*:\s*([^\n.;]+)",
        r"identified fault\s*:\s*([^\n.;]+)",
        r"diagnosis\s*:\s*([^\n.;]+)",
        r"operating condition\s*:\s*([^\n.;]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, str(report), flags=re.IGNORECASE)
        if match:
            return normalize_fault_name(match.group(1))

    return None


def count_prompt_echo_markers(report: str) -> int:
    low = str(report).lower()
    prompt_markers = [
        "retrieved technical knowledge",
        "retrieved context",
        "technical context block",
        "cnn-predicted fault class",
        "cnn confidence score",
        "report requirements:",
        "use only the information",
        "do not invent measurements",
        "finish the complete report",
        "the generated report must",
        "technical maintenance report for",
        "context 1",
        "retrieval score:",
    ]
    return sum(marker in low for marker in prompt_markers)


def contains_prompt_echo(report: str) -> bool:
    # Dois marcadores são exigidos para evitar falsos positivos.
    return count_prompt_echo_markers(report) >= 2


def contains_multiple_fault_declarations(report: str) -> bool:
    patterns = [
        r"predicted fault\s*:",
        r"fault type\s*:",
        r"diagnosed fault\s*:",
        r"identified fault\s*:",
        r"operating condition\s*:",
    ]
    total = sum(
        len(re.findall(pattern, str(report), flags=re.IGNORECASE))
        for pattern in patterns
    )
    return total > 2


def detect_base_invalid_reason(
    report: str,
    profile: str,
    minimum_chars: int = 120,
) -> Optional[str]:

    text = str(report).strip()
    low = text.lower()

    forbidden_placeholders = [
        "[insert",
        "insert date",
        "insert location",
        "insert equipment",
        "insert temperature",
        "[date]",
        "[location]",
        "[equipment",
    ]

    if len(text) < minimum_chars:
        return "too_short"
    if text in ["", "</s>", "<s>"]:
        return "empty_output"
    if "traceback" in low or "cuda" in low:
        return "runtime_text_leak"
    if any(item in low for item in forbidden_placeholders):
        return "placeholder_output"

    if profile == "mistral" and ("<unk>" in text or "ACHE" in text):
        return "malformed_output"

    if profile == "qwen" and text.count("!") > 20:
        return "repetitive_generation"

    if profile == "tinyllama":
        if (
            text.count("!") > 10
            or text.count("#") > 10
            or len(set(text.replace(" ", ""))) < 10
            or "- - -" in text
            or text.count("\n      -") > 20
            or text.count("\n        -") > 20
            or text.count("Fault Type:") > 3
            or text.count("Risk Level:") > 3
        ):
            return "repetitive_generation"

        if "\ndate:" in low or "\nlocation:" in low:
            return "template_artifact"

    return None


def audit_report(
    report: str,
    expected_fault: str,
    profile: str,
    minimum_chars: int,
    was_truncated: bool,
    input_was_truncated: bool,
) -> Dict[str, Any]:

    expected_normalized = normalize_fault_name(expected_fault)
    declared_fault = extract_declared_fault(report)

    # Ausência de declaração explícita não é considerada inconsistência.
    fault_consistent = (
        declared_fault is None
        or declared_fault == expected_normalized
    )

    prompt_echo_detected = contains_prompt_echo(report)
    multiple_fault_declarations = contains_multiple_fault_declarations(report)
    base_reason = detect_base_invalid_reason(
        report,
        profile,
        minimum_chars=minimum_chars,
    )

    if input_was_truncated:
        generation_issue = "input_truncation"
    elif was_truncated:
        generation_issue = "output_truncation"
    elif not fault_consistent:
        generation_issue = "class_inconsistency"
    elif prompt_echo_detected:
        generation_issue = "prompt_echo"
    elif multiple_fault_declarations:
        generation_issue = "multiple_fault_declarations"
    elif base_reason is not None:
        generation_issue = base_reason
    else:
        generation_issue = "valid"

    return {
        "expected_fault": expected_normalized,
        "declared_fault": declared_fault,
        "fault_consistent": bool(fault_consistent),
        "prompt_echo_detected": bool(prompt_echo_detected),
        "prompt_echo_marker_count": int(count_prompt_echo_markers(report)),
        "multiple_fault_declarations": bool(multiple_fault_declarations),
        "generation_issue": generation_issue,
        "valid_report": generation_issue == "valid",
    }


def apply_model_chat_template(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt.strip()}]

    if (
        hasattr(tokenizer, "apply_chat_template")
        and tokenizer.chat_template is not None
    ):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return prompt.strip()


def configure_seed(seed: int) -> None:
    set_seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def generate_report(
    prompt: str,
    config: Dict[str, Any],
    seed: int,
    expected_fault: str,
) -> Dict[str, Any]:

    configure_seed(seed)
    input_text = apply_model_chat_template(prompt)

    raw_inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=False,
        truncation=False,
    )

    raw_input_tokens = int(raw_inputs["input_ids"].shape[-1])
    max_input_tokens = int(config["max_input_tokens"])
    input_was_truncated = raw_input_tokens > max_input_tokens

    if input_was_truncated:
        raise RuntimeError(
            f"Prompt excedeu o limite final: "
            f"{raw_input_tokens} > {max_input_tokens} tokens. "
            "Nenhuma geração foi executada."
        )

    inputs = {
        key: value.to(model.device)
        for key, value in raw_inputs.items()
    }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    generation_kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs.get("attention_mask"),
        "max_new_tokens": int(config["max_new_tokens"]),
        "do_sample": bool(config["do_sample"]),
        "repetition_penalty": float(config["repetition_penalty"]),
        "pad_token_id": (
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        ),
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
        "return_dict_in_generate": True,
    }

    if generation_kwargs["do_sample"]:
        if config.get("temperature") is not None:
            generation_kwargs["temperature"] = float(config["temperature"])
        if config.get("top_p") is not None:
            generation_kwargs["top_p"] = float(config["top_p"])
        if config.get("top_k") is not None:
            generation_kwargs["top_k"] = int(config["top_k"])

    generation_kwargs = {
        key: value
        for key, value in generation_kwargs.items()
        if value is not None
    }

    start = time.perf_counter()
    with torch.inference_mode():
        generation_output = model.generate(**generation_kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency = time.perf_counter() - start

    sequences = generation_output.sequences
    generated_ids = sequences[0][raw_input_tokens:]

    output_tokens = int(generated_ids.shape[-1])
    eos_id = tokenizer.eos_token_id
    ended_with_eos = bool(
        eos_id is not None
        and (generated_ids == eos_id).any().item()
    )

    reached_token_limit = output_tokens >= int(config["max_new_tokens"])
    was_truncated = bool(reached_token_limit and not ended_with_eos)

    if ended_with_eos:
        finish_reason = "eos_token"
    elif reached_token_limit:
        finish_reason = "length"
    else:
        finish_reason = "other"

    report = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    peak_vram_gb = None
    if torch.cuda.is_available():
        peak_vram_gb = round(
            torch.cuda.max_memory_allocated() / 1024**3,
            3,
        )

    audit = audit_report(
        report=report,
        expected_fault=expected_fault,
        profile=config.get("invalid_rules", "generic"),
        minimum_chars=int(config.get("minimum_report_chars", 120)),
        was_truncated=was_truncated,
        input_was_truncated=input_was_truncated,
    )

    return {
        "report": report,
        **audit,
        "latency_seconds": latency,
        "input_tokens": raw_input_tokens,
        "raw_input_tokens": raw_input_tokens,
        "output_tokens": output_tokens,
        "peak_vram_gb": peak_vram_gb,
        "seed": seed,
        "finish_reason": finish_reason,
        "ended_with_eos": ended_with_eos,
        "reached_token_limit": reached_token_limit,
        "was_truncated": was_truncated,
        "input_was_truncated": input_was_truncated,
    }


## 12. Teste mínimo obrigatório antes dos 6 casos

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_model_key = MODELS_TO_RUN[0]
    smoke_config = {
        **GLOBAL_CONFIG,
        **MODEL_PROFILES[smoke_model_key],
    }

    tokenizer, model = load_slm(smoke_config)

    test_prompt = """
Predicted fault: Bearing Failure
CNN confidence: 98.7%

Retrieved evidence:
- Elevated temperature near the bearing housing.
- Possible lubrication degradation.

Generate a concise technical maintenance report with diagnosis, evidence, and recommended actions.
"""

    test_gen = generate_report(
        test_prompt,
        smoke_config,
        seed=smoke_config["base_seed"] - 1,
        expected_fault="Bearing Failure",
    )

    print(json.dumps(test_gen, indent=2, ensure_ascii=False))
    unload_slm()


## 13. Execução dos seis casos

In [ ]:
def safe_get_fault_type(doc):
    return doc.get("fault_type", doc.get("type", doc.get("id", "NA")))


def safe_get_section(doc):
    return doc.get("section", "full_entry")


def safe_get_score(doc):
    return doc.get("score", 0.0)


def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def run_experiment(
    cases: List[Dict[str, Any]],
    config: Dict[str, Any],
) -> pd.DataFrame:

    results = []

    for case_index, case in enumerate(
        tqdm(
            cases,
            desc=config.get("model_label", config.get("model_name")),
        )
    ):
        case_seed = int(config["base_seed"]) + case_index
        canonical_class = canonicalize_class(case["predicted_class"])

        retrieved = retrieve_context(
            predicted_class=canonical_class,
            confidence=case["confidence"],
            observed_evidence=case.get("observed_evidence", ""),
            top_k=config["rag_top_k"],
        )

        prompt, retrieved_context_text, context_budget_metadata = build_prompt(
            predicted_class=canonical_class,
            confidence=case["confidence"],
            retrieved_docs=retrieved,
            observed_evidence=case.get("observed_evidence", ""),
        )

        retrieved_context_json = json.dumps(
            retrieved,
            ensure_ascii=False,
        )

        try:
            gen = generate_report(
                prompt=prompt,
                config=config,
                seed=case_seed,
                expected_fault=canonical_class,
            )
            generation_error = None

        except Exception as exc:
            # O erro é registrado para não perder as demais classes/modelos.
            generation_error = f"{type(exc).__name__}: {exc}"
            gen = {
                "report": "",
                "expected_fault": normalize_fault_name(canonical_class),
                "declared_fault": None,
                "fault_consistent": False,
                "prompt_echo_detected": False,
                "prompt_echo_marker_count": 0,
                "multiple_fault_declarations": False,
                "generation_issue": "generation_error",
                "valid_report": False,
                "latency_seconds": np.nan,
                "input_tokens": np.nan,
                "raw_input_tokens": np.nan,
                "output_tokens": 0,
                "peak_vram_gb": (
                    round(torch.cuda.max_memory_allocated() / 1024**3, 3)
                    if torch.cuda.is_available()
                    else None
                ),
                "seed": case_seed,
                "finish_reason": "error",
                "ended_with_eos": False,
                "reached_token_limit": False,
                "was_truncated": False,
                "input_was_truncated": False,
            }

        row = {
            **case,
            "predicted_class_original": case["predicted_class"],
            "predicted_class": canonical_class,
            "model_key": config["model_key"],
            "model_name": config["model_name"],
            "model_label": config.get("model_label"),
            "loading_mode": config.get("loading_mode"),
            "experiment_version": config.get("experiment_version"),
            "temperature": config.get("temperature"),
            "top_p": config.get("top_p"),
            "top_k": config.get("top_k"),
            "do_sample": config.get("do_sample"),
            "repetition_penalty": config.get("repetition_penalty"),
            "max_new_tokens": config.get("max_new_tokens"),
            "max_input_tokens": config.get("max_input_tokens"),
            "rag_top_k": config["rag_top_k"],
            "retrieved_faults": " | ".join(
                safe_get_fault_type(d) for d in retrieved
            ),
            "retrieved_sections": " | ".join(
                safe_get_section(d) for d in retrieved
            ),
            "retrieval_scores": " | ".join(
                f"{safe_get_score(d):.4f}" for d in retrieved
            ),
            "retrieved_context": retrieved_context_text,
            "retrieved_context_json": retrieved_context_json,
            "context_budget_json": json.dumps(
                context_budget_metadata,
                ensure_ascii=False,
            ),
            "retrieved_context_sha256": stable_hash(retrieved_context_text),
            "prompt": prompt,
            "prompt_sha256": stable_hash(prompt),
            "context_was_trimmed": any(
                item["document_was_trimmed"]
                for item in context_budget_metadata["documents"]
            ),
            "generation_error": generation_error,
            **gen,
        }

        results.append(row)

    return pd.DataFrame(results)


def build_model_config(model_key: str) -> Dict[str, Any]:
    config = {
        **GLOBAL_CONFIG,
        **MODEL_PROFILES[model_key],
    }

    stem = f"slm_rag_results_{model_key}_final_v4"
    config["output_csv"] = str(results_dir / f"{stem}.csv")
    config["output_json"] = str(results_dir / f"{stem}.json")
    config["manifest_json"] = str(
        results_dir / f"manifest_{model_key}_final_v4.json"
    )
    return config


all_model_results = []
execution_status = []

for model_position, model_key in enumerate(MODELS_TO_RUN, start=1):
    print("\n" + "=" * 88)
    print(
        f"MODELO {model_position}/{len(MODELS_TO_RUN)}: "
        f"{MODEL_PROFILES[model_key]['model_label']}"
    )
    print("=" * 88)

    config = build_model_config(model_key)
    model_started_at = time.perf_counter()

    try:
        unload_slm()
        tokenizer, model = load_slm(config)

        model_df = run_experiment(
            cases=test_cases,
            config=config,
        )

        # Salva imediatamente após cada modelo.
        model_df.to_csv(
            config["output_csv"],
            index=False,
            encoding="utf-8",
        )

        with open(config["output_json"], "w", encoding="utf-8") as f:
            json.dump(
                model_df.to_dict(orient="records"),
                f,
                ensure_ascii=False,
                indent=2,
                default=str,
            )

        manifest = {
            "experiment_version": config["experiment_version"],
            "model_key": model_key,
            "model_name": config["model_name"],
            "model_label": config["model_label"],
            "loading_mode": config["loading_mode"],
            "generation_config": {
                key: config.get(key)
                for key in [
                    "max_new_tokens",
                    "do_sample",
                    "temperature",
                    "top_p",
                    "top_k",
                    "repetition_penalty",
                    "base_seed",
                    "max_input_tokens",
                    "prompt_safety_margin",
                ]
            },
            "rag_config": {
                "embedding_model": config["embedding_model"],
                "rag_top_k": config["rag_top_k"],
                "rag_json_path": config["rag_json_path"],
                "number_of_documents": len(documents),
            },
            "audit_protocol": {
                "class_consistency_check": True,
                "prompt_echo_check": True,
                "multiple_fault_declarations_check": True,
                "truncation_check": True,
                "invalid_outputs_are_preserved": True,
                "automatic_regeneration": False,
            },
            "environment": {
                "python": platform.python_version(),
                "torch": torch.__version__,
                "cuda_available": torch.cuda.is_available(),
                "gpu": (
                    torch.cuda.get_device_name(0)
                    if torch.cuda.is_available()
                    else None
                ),
            },
            "files": {
                "csv": config["output_csv"],
                "json": config["output_json"],
            },
            "counts": {
                "total": int(len(model_df)),
                "valid": int(model_df["valid_report"].sum()),
                "invalid": int((~model_df["valid_report"]).sum()),
                "truncated": int(model_df["was_truncated"].sum()),
                "class_inconsistent": int(
                    (~model_df["fault_consistent"]).sum()
                ),
                "prompt_echo": int(
                    model_df["prompt_echo_detected"].sum()
                ),
            },
        }

        with open(config["manifest_json"], "w", encoding="utf-8") as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)

        all_model_results.append(model_df)

        execution_status.append({
            "model_key": model_key,
            "model_label": config["model_label"],
            "status": "completed",
            "rows": len(model_df),
            "valid_reports": int(model_df["valid_report"].sum()),
            "invalid_reports": int((~model_df["valid_report"]).sum()),
            "elapsed_seconds": time.perf_counter() - model_started_at,
            "error": None,
        })

        print("CSV salvo em:", config["output_csv"])
        print("JSON salvo em:", config["output_json"])
        print("Manifesto salvo em:", config["manifest_json"])

    except Exception as exc:
        error_text = f"{type(exc).__name__}: {exc}"
        execution_status.append({
            "model_key": model_key,
            "model_label": config["model_label"],
            "status": "failed",
            "rows": 0,
            "valid_reports": 0,
            "invalid_reports": 0,
            "elapsed_seconds": time.perf_counter() - model_started_at,
            "error": error_text,
        })

        print(f"ERRO NO MODELO {model_key}: {error_text}")

        if GLOBAL_CONFIG.get("stop_on_model_error", False):
            unload_slm()
            raise

    finally:
        unload_slm()


if not all_model_results:
    raise RuntimeError("Nenhum modelo produziu resultados.")

results_df = pd.concat(
    all_model_results,
    ignore_index=True,
)

consolidated_csv = results_dir / "slm_rag_results_all_models_final_v4.csv"
consolidated_json = results_dir / "slm_rag_results_all_models_final_v4.json"
status_csv = results_dir / "execution_status_all_models_final_v4.csv"

results_df.to_csv(
    consolidated_csv,
    index=False,
    encoding="utf-8",
)

with open(consolidated_json, "w", encoding="utf-8") as f:
    json.dump(
        results_df.to_dict(orient="records"),
        f,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

execution_status_df = pd.DataFrame(execution_status)
execution_status_df.to_csv(
    status_csv,
    index=False,
    encoding="utf-8",
)

print("\n" + "=" * 88)
print("EXECUÇÃO MULTIMODELO CONCLUÍDA")
print("=" * 88)
print("CSV consolidado:", consolidated_csv)
print("JSON consolidado:", consolidated_json)
print("Status:", status_csv)

display(execution_status_df)


## 14. Visualizar relatórios e auditoria


In [ ]:
display_columns = [
    "model_label",
    "sample_id",
    "predicted_class",
    "expected_fault",
    "declared_fault",
    "fault_consistent",
    "prompt_echo_detected",
    "generation_issue",
    "valid_report",
    "latency_seconds",
    "output_tokens",
]

display(results_df[display_columns])

# Exemplo: primeiro relatório gerado.
i = 0
print("\n" + "=" * 80)
print("Modelo:", results_df.loc[i, "model_label"])
print("Classe prevista:", results_df.loc[i, "predicted_class"])
print("Auditoria:", results_df.loc[i, "generation_issue"])
print("-" * 80)
print(results_df.loc[i, "report"])


## 15. Grid exploratório antigo

Desativado na rodada final. Os parâmetros de geração já estão congelados e padronizados.


In [ ]:

# ============================================================
# GRID EXPLORATÓRIO ANTIGO — NÃO EXECUTAR NA RODADA FINAL
# ============================================================
# A rodada final já possui parâmetros congelados e iguais entre os modelos.
# Esta célula é mantida apenas para documentação histórica.

RUN_GRID = False

if RUN_GRID:
    raise RuntimeError(
        "O grid exploratório foi desativado para a rodada final controlada."
    )


In [ ]:
# ============================================================
# GRID EXPLORATÓRIO ANTIGO — DESATIVADO
# ============================================================
# A rodada final usa parâmetros congelados e executa os quatro modelos.
# Esta célula é mantida apenas como registro histórico.

RUN_GRID = False

if RUN_GRID:
    raise RuntimeError(
        "O grid está desativado na rodada final multimodelo."
    )


## 16. Métricas locais opcionais

A avaliação científica final deve ser feita no notebook consolidado de métricas, usando os CSVs finais validados.


In [ ]:
# def add_metrics_if_reference_exists(
#     df: pd.DataFrame,
#     generated_col: str = "report",
#     reference_col: str = "reference_report"
# ) -> pd.DataFrame:
#     df = df.copy()

#     if reference_col not in df.columns:
#         print(f"Coluna '{reference_col}' não encontrada. Métricas textuais não calculadas.")
#         return df

#     from sklearn.metrics.pairwise import cosine_similarity
#     import evaluate

#     metric_embedding_model = SentenceTransformer(GLOBAL_CONFIG["embedding_model"])
#     bleu = evaluate.load("bleu")
#     rouge = evaluate.load("rouge")

#     cosine_scores = []
#     bleu_scores = []
#     rouge_l_scores = []

#     for _, row in tqdm(df.iterrows(), total=len(df)):
#         generated = str(row[generated_col])
#         reference = str(row[reference_col])

#         emb = metric_embedding_model.encode(
#             [generated, reference],
#             convert_to_numpy=True,
#             normalize_embeddings=True
#         )
#         cosine_scores.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))

#         bleu_scores.append(float(bleu.compute(predictions=[generated], references=[[reference]])["bleu"]))
#         rouge_l_scores.append(float(rouge.compute(predictions=[generated], references=[reference])["rougeL"]))

#     df["semantic_cosine_similarity"] = cosine_scores
#     df["bleu"] = bleu_scores
#     df["rougeL"] = rouge_l_scores
#     return df


# results_df = add_metrics_if_reference_exists(results_df)
# results_df.to_csv(GLOBAL_CONFIG["output_csv"], index=False, encoding="utf-8")
# print("CSV atualizado:", GLOBAL_CONFIG["output_csv"])

## 17. Consolidação e comparação operacional

Os CSVs individuais e o CSV consolidado já foram gerados na execução multimodelo.


In [ ]:
# from pathlib import Path

# csv_files = sorted(Path(GLOBAL_CONFIG["results_dir"]).glob("slm_rag_results_*.csv"))
# print("CSVs encontrados:", [p.name for p in csv_files])

# if csv_files:
#     comparison_df = pd.concat([pd.read_csv(p) for p in csv_files], ignore_index=True)

#     summary_cols = {
#         "latency_seconds": "mean",
#         "input_tokens": "mean",
#         "output_tokens": "mean",
#         "peak_vram_gb": "max",
#         "valid_report": "mean",
#     }

#     for optional in ["semantic_cosine_similarity", "bleu", "rougeL"]:
#         if optional in comparison_df.columns:
#             summary_cols[optional] = "mean"

#     summary_df = comparison_df.groupby("model_label", as_index=False).agg(summary_cols)
#     display(summary_df)
# else:
#     print("Nenhum CSV encontrado ainda.")

In [ ]:
# ============================================================
# RESUMO FINAL DA EXECUÇÃO MULTIMODELO
# ============================================================

summary_columns = [
    "model_label",
    "sample_id",
    "predicted_class",
    "confidence",
    "temperature",
    "top_p",
    "max_new_tokens",
    "input_tokens",
    "output_tokens",
    "latency_seconds",
    "peak_vram_gb",
    "finish_reason",
    "input_was_truncated",
    "was_truncated",
    "fault_consistent",
    "prompt_echo_detected",
    "generation_issue",
    "valid_report",
]

execution_summary = results_df[
    [c for c in summary_columns if c in results_df.columns]
].copy()

display(execution_summary)

print("\nRESUMO POR MODELO")

model_summary = (
    results_df
    .groupby(["model_key", "model_label"], dropna=False)
    .agg(
        total_reports=("report", "size"),
        valid_reports=("valid_report", "sum"),
        valid_rate=("valid_report", "mean"),
        class_consistent_rate=("fault_consistent", "mean"),
        prompt_echo_count=("prompt_echo_detected", "sum"),
        truncated_count=("was_truncated", "sum"),
        latency_mean=("latency_seconds", "mean"),
        latency_std=("latency_seconds", "std"),
        output_tokens_mean=("output_tokens", "mean"),
        peak_vram_max=("peak_vram_gb", "max"),
    )
    .reset_index()
)

model_summary["valid_rate"] = (
    model_summary["valid_rate"] * 100
).round(2)

model_summary["class_consistent_rate"] = (
    model_summary["class_consistent_rate"] * 100
).round(2)

display(model_summary.round(3))

print("\nTIPOS DE OCORRÊNCIA POR MODELO")

issue_summary = (
    results_df
    .groupby(
        ["model_key", "model_label", "generation_issue"],
        dropna=False,
    )
    .size()
    .reset_index(name="count")
)

display(issue_summary)

model_summary_path = results_dir / "summary_by_model_final_v4.csv"
issue_summary_path = results_dir / "generation_issues_final_v4.csv"

model_summary.to_csv(
    model_summary_path,
    index=False,
    encoding="utf-8",
)
issue_summary.to_csv(
    issue_summary_path,
    index=False,
    encoding="utf-8",
)

print("Resumo salvo em:", model_summary_path)
print("Ocorrências salvas em:", issue_summary_path)


## 18. Entrada futura da CNN

Nesta versão, os casos ainda entram como `predicted_class` e `confidence`, preservando a comparação entre SLMs. Depois o classificador com 5-fold poderá preencher automaticamente esses dois campos.

In [ ]:
# Fluxo definitivo:
# holdout_test_predictions.csv
#   -> seleção automática de uma amostra correta por classe
#   -> classe prevista + confiança reais da CNN
#   -> recuperação top-k=3 na base RAG original
#   -> limitação determinística do contexto
#   -> execução sequencial de Mistral, Qwen, TinyLlama e Zephyr
#   -> descarregamento do modelo entre rodadas
#   -> auditoria de classe, eco do prompt, repetição e truncamento
#   -> preservação das falhas generativas sem regeneração automática
#   -> CSV/JSON individual + consolidado + manifestos
